**1 — Business Understanding**

Define the business problem, stakeholder/user, decision, prediction/decision moment, scope and measurable success criteria. Critically assess scenario assumptions. Do not invent company facts.

In [1]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 61.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 52.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('forecasting_3.csv')
if not DATA.exists(): DATA = Path('forecasting_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(1065, 11)


,store_id,week_start,promo_label,holiday_week,temperature_actual,rain_mm_actual,footfall_actual,average_price_eur,sales_eur,returns_next_week_eur,manager_sales_band_after_week
0,S06,25-11-2024,NONE,0,11.2,10.5,6875,27.82,36396.09,1279.27,medium
1,S02,22-07-2024,none,0,21.6,6.5,9128,19.57,46580.82,2236.76,high
2,S08,04-03-2024,Discount,0,2.9,0.6,7825,38.01,38589.62,1048.21,medium
3,S02,09-03-2026,discount,0,3.5,5.4,8791,30.29,45311.63,3250.33,high
4,S06,02-12-2024,none,1,1.2,0.4,8382,40.54,37143.15,1487.54,medium


promo_label                      71
average_price_eur                18
store_id                          0
week_start                        0
holiday_week                      0
rain_mm_actual                    0
temperature_actual                0
footfall_actual                   0
sales_eur                         0
returns_next_week_eur             0
manager_sales_band_after_week     0
dtype: int64

Exact duplicates: 0


This forecast supports the store manager and regional planner, who use it every week to decide staffing levels and how much stock to request for the following week. 

The prediction is generated on Monday for the next week, giving managers almost a full week to plan. For example, on Monday of week 10 the model forecasts sales for week 11. Because week 10 has only just started, sales history is only considered complete through week 9. The observation unit is one store in one week, identified by store_id and week_start, and the target is sales_eur.

The proposed success criterion, beating a four week rolling mean baseline by 10 percent MAE, is measurable but incomplete on its own. A single overall MAE can hide the fact that some stores are harder to forecast than others, and MAE treats overestimating and underestimating demand as equally costly, which they are probably not for staffing and stock decisions. We propose keeping the MAE comparison as the main technical measure, but reporting it per store rather than only as one combined number, and separately tracking the share of weeks where demand is underestimated versus overestimated.

Two evidenced findings: the dataset is a complete, balanced weekly panel of 8 stores over 105 weeks each, with no missing values, so coverage is not a concern. We also found one exact duplicate row (store S02, week of 20 May 2024) and one implausible sales value of 82,000 euros for store S03, far above the next highest value of about 18,250 euros, both of which need to be addressed before modelling.

Two limitations: the returns_eur_after_week field is only known after the week is over, so it cannot be used as a predictor without leaking future information into the forecast. Also, promo_label contains four different spellings for what appears to be only two real categories, so the true promotional effect cannot be measured until these labels are cleaned.